<a href="https://colab.research.google.com/github/Saadiphone/thalf/blob/main/TRELLIS_jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# خلية واحدة فقط — بعد Restart session
# ============================================================
import os, sys
os.environ['CUMM_DISABLE_JIT'] = '1'
os.environ['SPCONV_DISABLE_JIT'] = '1'
os.environ['ATTN_BACKEND'] = 'xformers'
os.environ['SPCONV_ALGO'] = 'native'

# إصلاح الملف قبل أي import
init_file = "/content/TRELLIS/trellis/modules/sparse/__init__.py"
with open(init_file, 'r') as f:
    content = f.read()
content = content.replace("BACKEND = 'torchsparse'", "BACKEND = 'spconv'")
with open(init_file, 'w') as f:
    f.write(content)

# مسح __pycache__
os.system("find /content/TRELLIS -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null")
os.system("find /content/TRELLIS -name '*.pyc' -delete 2>/dev/null")

sys.path.insert(0, '/content/TRELLIS')
os.chdir('/content/TRELLIS')

# الآن import
import numpy as np
import torch
import imageio
from typing import *
from PIL import Image
from easydict import EasyDict as edict
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.representations import Gaussian, MeshExtractResult
from trellis.utils import render_utils, postprocessing_utils

MAX_SEED = np.iinfo(np.int32).max
TMP_DIR = "/content"

def preprocess_image(image_path: str) -> Tuple[str, Image.Image]:
    trial_id = "trellis-tost"
    image = Image.open(image_path).convert("RGBA")
    processed_image = pipeline.preprocess_image(image)
    processed_image.save(f"{TMP_DIR}/{trial_id}.png")
    return trial_id, processed_image

def pack_state(gs: Gaussian, mesh: MeshExtractResult, trial_id: str) -> dict:
    return {
        'gaussian': {
            **gs.init_params,
            '_xyz': gs._xyz.cpu().numpy(),
            '_features_dc': gs._features_dc.cpu().numpy(),
            '_scaling': gs._scaling.cpu().numpy(),
            '_rotation': gs._rotation.cpu().numpy(),
            '_opacity': gs._opacity.cpu().numpy(),
        },
        'mesh': {
            'vertices': mesh.vertices.cpu().numpy(),
            'faces': mesh.faces.cpu().numpy(),
        },
        'trial_id': trial_id,
    }

def unpack_state(state: dict) -> Tuple[Gaussian, edict, str]:
    gs = Gaussian(
        aabb=state['gaussian']['aabb'],
        sh_degree=state['gaussian']['sh_degree'],
        mininum_kernel_size=state['gaussian']['mininum_kernel_size'],
        scaling_bias=state['gaussian']['scaling_bias'],
        opacity_bias=state['gaussian']['opacity_bias'],
        scaling_activation=state['gaussian']['scaling_activation'],
    )
    gs._xyz = torch.tensor(state['gaussian']['_xyz'], device='cuda')
    gs._features_dc = torch.tensor(state['gaussian']['_features_dc'], device='cuda')
    gs._scaling = torch.tensor(state['gaussian']['_scaling'], device='cuda')
    gs._rotation = torch.tensor(state['gaussian']['_rotation'], device='cuda')
    gs._opacity = torch.tensor(state['gaussian']['_opacity'], device='cuda')
    mesh = edict(
        vertices=torch.tensor(state['mesh']['vertices'], device='cuda'),
        faces=torch.tensor(state['mesh']['faces'], device='cuda'),
    )
    return gs, mesh, state['trial_id']

def image_to_3d(image_path: str, seed: int = 0, randomize_seed: bool = True,
                ss_guidance_strength: float = 7.5, ss_sampling_steps: int = 12,
                slat_guidance_strength: float = 3.0, slat_sampling_steps: int = 12) -> Tuple[dict, str]:
    trial_id, _ = preprocess_image(image_path)
    if randomize_seed:
        seed = np.random.randint(0, MAX_SEED)
    outputs = pipeline.run(
        Image.open(f"{TMP_DIR}/{trial_id}.png"),
        seed=seed, formats=["gaussian", "mesh"], preprocess_image=False,
        sparse_structure_sampler_params={"steps": ss_sampling_steps, "cfg_strength": ss_guidance_strength},
        slat_sampler_params={"steps": slat_sampling_steps, "cfg_strength": slat_guidance_strength},
    )
    video = render_utils.render_video(outputs['gaussian'][0], num_frames=120)['color']
    video_geo = render_utils.render_video(outputs['mesh'][0], num_frames=120)['normal']
    video = [np.concatenate([video[i], video_geo[i]], axis=1) for i in range(len(video))]
    video_path = f"{TMP_DIR}/trellis-tost.mp4"
    imageio.mimsave(video_path, video, fps=15)
    state = pack_state(outputs['gaussian'][0], outputs['mesh'][0], "trellis-tost")
    return state, video_path

def extract_glb(state: dict, mesh_simplify: float = 0.95, texture_size: int = 1024) -> str:
    gs, mesh, trial_id = unpack_state(state)
    glb = postprocessing_utils.to_glb(gs, mesh, simplify=mesh_simplify, texture_size=texture_size, verbose=False)
    glb_path = f"{TMP_DIR}/{trial_id}.glb"
    glb.export(glb_path)
    return glb_path

with torch.inference_mode():
    pipeline = TrellisImageTo3DPipeline.from_pretrained("/content/model")
    pipeline.cuda()

print("✅ الموديل جاهز للاستخدام!")

[SPARSE] Backend: spconv, Attention: xformers


/content/TRELLIS/trellis/models/sparse_structure_vae.py:103: SyntaxWarning: invalid escape sequence '\m'
  Encoder for Sparse Structure (\mathcal{E}_S in the paper Sec. 3.3).
/content/TRELLIS/trellis/models/sparse_structure_vae.py:212: SyntaxWarning: invalid escape sequence '\m'
  Decoder for Sparse Structure (\mathcal{D}_S in the paper Sec. 3.3).


[SPARSE][CONV] spconv algo: native
[ATTENTION] Using backend: xformers


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


✅ الموديل جاهز للاستخدام!


In [5]:
# الخطوة 1: تثبيت spconv و cumm
import os
os.environ['CUMM_DISABLE_JIT'] = '1'
os.environ['SPCONV_DISABLE_JIT'] = '1'

!pip install cumm-cu124 spconv-cu124

# تحقق
!python -c "import spconv; print('✅ spconv:', spconv.__version__)"
!python -c "import cumm; print('✅ cumm:', cumm.__version__)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.0/70.0 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.6/27.6 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 15.1 MB/s eta 0:00:00
✅ spconv: 2.3.8
✅ cumm: 0.7.11


In [4]:
!pip list | grep -i -E "spconv|cumm"
!python -c "import spconv; print('spconv OK:', spconv.__version__)" 2>&1
!python -c "import cumm; print('cumm OK:', cumm.__version__)" 2>&1

Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'spconv'
Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'cumm'


In [3]:
# ============================================================
# الإصلاح — شغّل هذا فقط
# ============================================================

# 1. إرجاع الـ backend لـ spconv
!sed -i "s/BACKEND = 'torchsparse'/BACKEND = 'spconv'/" /content/TRELLIS/trellis/modules/sparse/__init__.py

# 2. تحقق إن spconv مثبت
!pip list | grep -i spconv
!pip list | grep -i cumm

# 3. تحقق إن الـ backend رجع spconv
!head -5 /content/TRELLIS/trellis/modules/sparse/__init__.py

from typing import *

BACKEND = 'spconv' 
DEBUG = False
ATTN = 'flash_attn'


In [3]:
# ============================================================
# إصلاح شامل - شغّل في Runtime نظيف
# ============================================================

# الخطوة 1: تعطيل JIT لـ cumm/spconv قبل أي شي
import os
os.environ['CUMM_DISABLE_JIT'] = '1'
os.environ['SPCONV_DISABLE_JIT'] = '1'
os.environ['ATTN_BACKEND'] = 'xformers'
os.environ['SPCONV_ALGO'] = 'native'

# الخطوة 2: تحقق إن الملفات موجودة
!echo "=== التحقق من pipeline.json ==="
!cat /content/model/pipeline.json

!echo ""
!echo "=== التحقق من ملفات الموديل ==="
!ls -la /content/model/ckpts/

=== التحقق من pipeline.json ===
{
    "name": "TrellisImageTo3DPipeline",
    "args": {
        "models": {
            "sparse_structure_decoder": "ckpts/ss_dec_conv3d_16l8_fp16",
            "sparse_structure_flow_model": "ckpts/ss_flow_img_dit_L_16l8_fp16",
            "slat_decoder_gs": "ckpts/slat_dec_gs_swin8_B_64l8gs32_fp16",
            "slat_decoder_rf": "ckpts/slat_dec_rf_swin8_B_64l8r16_fp16",
            "slat_decoder_mesh": "ckpts/slat_dec_mesh_swin8_B_64l8m256c_fp16",
            "slat_flow_model": "ckpts/slat_flow_img_dit_L_64l8p2_fp16"
        },
        "sparse_structure_sampler": {
            "name": "FlowEulerGuidanceIntervalSampler",
            "args": {
                "sigma_min": 1e-5
            },
            "params": {
                "steps": 25,
                "cfg_strength": 5.0,
                "cfg_interval": [0.5, 1.0],
                "rescale_t": 3.0
            }
        },
        "slat_sampler": {
            "name": "FlowEulerGuidanceIntervalSa

In [1]:
# ============================================================
# الخلية 1: تثبيت + إعداد كامل (النسخة النهائية)
# ============================================================
import os
os.environ['SPARSE_BACKEND'] = 'torchsparse'
os.environ['ATTN_BACKEND'] = 'xformers'
os.environ['SPCONV_ALGO'] = 'native'

!apt install aria2 -qqy

# --- 1. PyTorch + CUDA 12.4 ---
!pip install torch==2.5.0+cu124 torchvision==0.20.0+cu124 torchaudio==2.5.0+cu124 --extra-index-url https://download.pytorch.org/whl/cu124

# --- 2. xformers ---
!pip install xformers==0.0.28.post2

# --- 3. ninja (مطلوب لبناء الحزم) ---
!pip install ninja

# --- 4. استنساخ TRELLIS ---
%cd /content
!git clone --recursive https://github.com/Microsoft/TRELLIS
%cd /content/TRELLIS

# --- 5. تبديل الـ backend من spconv إلى torchsparse ---
!sed -i "s/BACKEND = 'spconv'/BACKEND = 'torchsparse'/" /content/TRELLIS/trellis/modules/sparse/__init__.py

# --- 6. تثبيت torchsparse ---
!python -c "$(curl -fsSL https://raw.githubusercontent.com/mit-han-lab/torchsparse/master/install.py)"

# --- 7. الحزم الأساسية ---
!pip install easydict rembg onnxruntime onnxruntime-gpu numpy==2.0.0 plyfile huggingface-hub safetensors
!pip install open3d
!pip install trimesh xatlas pyvista pymeshfix igraph
!pip install opencv-python imageio imageio-ffmpeg ffmpeg-python av

# --- 8. nvdiffrast من المصدر ---
%cd /content
!rm -rf nvdiffrast
!git clone https://github.com/NVlabs/nvdiffrast.git
%cd /content/nvdiffrast
!pip install --no-build-isolation .
%cd /content/TRELLIS

# --- 9. kaolin من NVIDIA ---
!pip install kaolin==0.18.0 -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.5.0_cu124.html

# --- 10. diso ---
!pip install diso

# --- 11. utils3d ---
!pip install https://github.com/camenduru/wheels/releases/download/3090/utils3d-0.0.2-py3-none-any.whl

# --- 12. diff_gaussian_rasterization ---
!pip install --no-build-isolation git+https://github.com/graphdeco-inria/diff-gaussian-rasterization

# --- 13. تحميل ملفات الموديل ---
!mkdir -p /content/model/ckpts
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/JeffreyXiang/TRELLIS-image-large/raw/main/pipeline.json -d /content/model -o pipeline.json
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/JeffreyXiang/TRELLIS-image-large/raw/main/ckpts/slat_dec_gs_swin8_B_64l8gs32_fp16.json -d /content/model/ckpts -o slat_dec_gs_swin8_B_64l8gs32_fp16.json
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/JeffreyXiang/TRELLIS-image-large/resolve/main/ckpts/slat_dec_gs_swin8_B_64l8gs32_fp16.safetensors -d /content/model/ckpts -o slat_dec_gs_swin8_B_64l8gs32_fp16.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/JeffreyXiang/TRELLIS-image-large/raw/main/ckpts/slat_dec_mesh_swin8_B_64l8m256c_fp16.json -d /content/model/ckpts -o slat_dec_mesh_swin8_B_64l8m256c_fp16.json
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/JeffreyXiang/TRELLIS-image-large/resolve/main/ckpts/slat_dec_mesh_swin8_B_64l8m256c_fp16.safetensors -d /content/model/ckpts -o slat_dec_mesh_swin8_B_64l8m256c_fp16.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/JeffreyXiang/TRELLIS-image-large/raw/main/ckpts/slat_dec_rf_swin8_B_64l8r16_fp16.json -d /content/model/ckpts -o slat_dec_rf_swin8_B_64l8r16_fp16.json
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/JeffreyXiang/TRELLIS-image-large/resolve/main/ckpts/slat_dec_rf_swin8_B_64l8r16_fp16.safetensors -d /content/model/ckpts -o slat_dec_rf_swin8_B_64l8r16_fp16.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/JeffreyXiang/TRELLIS-image-large/raw/main/ckpts/slat_enc_swin8_B_64l8_fp16.json -d /content/model/ckpts -o slat_enc_swin8_B_64l8_fp16.json
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/JeffreyXiang/TRELLIS-image-large/resolve/main/ckpts/slat_enc_swin8_B_64l8_fp16.safetensors -d /content/model/ckpts -o slat_enc_swin8_B_64l8_fp16.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/JeffreyXiang/TRELLIS-image-large/raw/main/ckpts/slat_flow_img_dit_L_64l8p2_fp16.json -d /content/model/ckpts -o slat_flow_img_dit_L_64l8p2_fp16.json
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/JeffreyXiang/TRELLIS-image-large/resolve/main/ckpts/slat_flow_img_dit_L_64l8p2_fp16.safetensors -d /content/model/ckpts -o slat_flow_img_dit_L_64l8p2_fp16.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/JeffreyXiang/TRELLIS-image-large/raw/main/ckpts/ss_dec_conv3d_16l8_fp16.json -d /content/model/ckpts -o ss_dec_conv3d_16l8_fp16.json
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/JeffreyXiang/TRELLIS-image-large/resolve/main/ckpts/ss_dec_conv3d_16l8_fp16.safetensors -d /content/model/ckpts -o ss_dec_conv3d_16l8_fp16.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/JeffreyXiang/TRELLIS-image-large/raw/main/ckpts/ss_enc_conv3d_16l8_fp16.json -d /content/model/ckpts -o ss_enc_conv3d_16l8_fp16.json
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/JeffreyXiang/TRELLIS-image-large/resolve/main/ckpts/ss_enc_conv3d_16l8_fp16.safetensors -d /content/model/ckpts -o ss_enc_conv3d_16l8_fp16.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/JeffreyXiang/TRELLIS-image-large/raw/main/ckpts/ss_flow_img_dit_L_16l8_fp16.json -d /content/model/ckpts -o ss_flow_img_dit_L_16l8_fp16.json
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/JeffreyXiang/TRELLIS-image-large/resolve/main/ckpts/ss_flow_img_dit_L_16l8_fp16.safetensors -d /content/model/ckpts -o ss_flow_img_dit_L_16l8_fp16.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://github.com/facebookresearch/dinov2/zipball/main -d /root/.cache/torch/hub -o main.zip
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_reg4_pretrain.pth -d /root/.cache/torch/hub/checkpoints -o dinov2_vitl14_reg4_pretrain.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://github.com/danielgatis/rembg/releases/download/v0.0.0/u2net.onnx -d /root/.u2net -o u2net.onnx

# --- 14. التحقق ---
!cat /content/TRELLIS/trellis/modules/sparse/__init__.py | head -5
!ls /content/model/ckpts/ | head -5

print("\n✅ التثبيت اكتمل!")

aria2 is already the newest version (1.36.0-1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu124
/content
fatal: destination path 'TRELLIS' already exists and is not an empty directory.
/content/TRELLIS
Traceback (most recent call last):
  File "<string>", line 40, in <module>
KeyError: 'torch25'
/content
Cloning into 'nvdiffrast'...
remote: Enumerating objects: 523, done.
remote: Counting objects: 100% (268/268), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 523 (delta 201), reused 190 (delta 190), pack-reused 255 (from 1)
Receiving objects: 100% (523/523), 11.12 MiB | 16.74 MiB/s, done.
Resolving deltas: 100% (285/285), done.
/content/nvdiffrast
Processing /content/nvdiffrast
  Preparing metadata (pyproject.toml) ... done
  Created wheel for nvdiffrast: filename=nvdiffrast-0.4.0-cp312-cp312-linux_x86_64.whl size=15219090 sha256=fe3346eb1c24934fced52bdc7d0677e23934

In [1]:
# ============================================================
# الإصلاح الشامل والنهائي
# ============================================================
import os
os.environ['CUMM_DISABLE_JIT'] = '1'
os.environ['SPCONV_DISABLE_JIT'] = '1'
os.environ['ATTN_BACKEND'] = 'xformers'
os.environ['SPCONV_ALGO'] = 'native'

# ============================================================
# الخطوة 1: إصلاح بنية مجلدات الموديل
# كل checkpoint يحتاج مجلد خاص فيه .json و .safetensors
# ============================================================
import shutil, glob

ckpts_dir = "/content/model/ckpts"
for json_file in glob.glob(f"{ckpts_dir}/*.json"):
    name = os.path.splitext(os.path.basename(json_file))[0]
    folder = f"{ckpts_dir}/{name}"
    os.makedirs(folder, exist_ok=True)
    # نقل .json إلى المجلد باسم .json
    shutil.copy2(json_file, f"{folder}/.json")
    # نقل .safetensors إلى المجلد
    st_file = json_file.replace('.json', '.safetensors')
    if os.path.exists(st_file):
        shutil.copy2(st_file, f"{folder}/model.safetensors")

print("✅ الخطوة 1: بنية المجلدات تم إصلاحها")
!ls /content/model/ckpts/slat_dec_mesh_swin8_B_64l8m256c_fp16/

# ============================================================
# الخطوة 2: إصلاح spconv - إنشاء symlink للـ headers الناقصة
# ============================================================
import subprocess

# نجد مكان cumm headers
cumm_path = "/usr/local/lib/python3.12/dist-packages/cumm"
include_path = f"{cumm_path}/include"

if os.path.exists(include_path):
    # إنشاء symlink لـ tensorview headers في مكان الـ build
    build_include = f"{cumm_path}/build/core_cc/include"
    if os.path.exists(build_include):
        src = include_path
        dst = f"{build_include}/tensorview"
        if not os.path.exists(dst):
            os.symlink(f"{include_path}/tensorview", dst)
            print(f"✅ تم ربط tensorview headers")

# ============================================================
# الخطوة 3: اختبار spconv
# ============================================================
try:
    import spconv
    print(f"✅ الخطوة 3: spconv يعمل! version: {spconv.__version__}")
except Exception as e:
    print(f"⚠️ spconv لا يعمل: {e}")
    print("جاري تثبيت torchsparse كبديل...")
    os.system("pip install --quiet git+https://github.com/mit-han-lab/torchsparse.git")
    # تغيير الـ backend
    os.environ['SPARSE_BACKEND'] = 'torchsparse'
    init_file = "/content/TRELLIS/trellis/modules/sparse/__init__.py"
    with open(init_file, 'r') as f:
        content = f.read()
    content = content.replace("BACKEND = 'spconv'", "BACKEND = 'torchsparse'")
    with open(init_file, 'w') as f:
        f.write(content)
    print("✅ تم التبديل إلى torchsparse")

✅ الخطوة 1: بنية المجلدات تم إصلاحها
model.safetensors
⚠️ spconv لا يعمل: No module named 'spconv'
جاري تثبيت torchsparse كبديل...
✅ تم التبديل إلى torchsparse


In [6]:
import os
os.environ['CUMM_DISABLE_JIT'] = '1'
os.environ['SPCONV_DISABLE_JIT'] = '1'
os.environ['ATTN_BACKEND'] = 'xformers'
os.environ['SPCONV_ALGO'] = 'native'

%cd /content/TRELLIS

import numpy as np
import torch
import imageio
from typing import *
from PIL import Image
from easydict import EasyDict as edict
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.representations import Gaussian, MeshExtractResult
from trellis.utils import render_utils, postprocessing_utils

MAX_SEED = np.iinfo(np.int32).max
TMP_DIR = "/content"

def preprocess_image(image_path: str) -> Tuple[str, Image.Image]:
    trial_id = "trellis-tost"
    image = Image.open(image_path).convert("RGBA")
    processed_image = pipeline.preprocess_image(image)
    processed_image.save(f"{TMP_DIR}/{trial_id}.png")
    return trial_id, processed_image

def pack_state(gs: Gaussian, mesh: MeshExtractResult, trial_id: str) -> dict:
    return {
        'gaussian': {
            **gs.init_params,
            '_xyz': gs._xyz.cpu().numpy(),
            '_features_dc': gs._features_dc.cpu().numpy(),
            '_scaling': gs._scaling.cpu().numpy(),
            '_rotation': gs._rotation.cpu().numpy(),
            '_opacity': gs._opacity.cpu().numpy(),
        },
        'mesh': {
            'vertices': mesh.vertices.cpu().numpy(),
            'faces': mesh.faces.cpu().numpy(),
        },
        'trial_id': trial_id,
    }

def unpack_state(state: dict) -> Tuple[Gaussian, edict, str]:
    gs = Gaussian(
        aabb=state['gaussian']['aabb'],
        sh_degree=state['gaussian']['sh_degree'],
        mininum_kernel_size=state['gaussian']['mininum_kernel_size'],
        scaling_bias=state['gaussian']['scaling_bias'],
        opacity_bias=state['gaussian']['opacity_bias'],
        scaling_activation=state['gaussian']['scaling_activation'],
    )
    gs._xyz = torch.tensor(state['gaussian']['_xyz'], device='cuda')
    gs._features_dc = torch.tensor(state['gaussian']['_features_dc'], device='cuda')
    gs._scaling = torch.tensor(state['gaussian']['_scaling'], device='cuda')
    gs._rotation = torch.tensor(state['gaussian']['_rotation'], device='cuda')
    gs._opacity = torch.tensor(state['gaussian']['_opacity'], device='cuda')
    mesh = edict(
        vertices=torch.tensor(state['mesh']['vertices'], device='cuda'),
        faces=torch.tensor(state['mesh']['faces'], device='cuda'),
    )
    return gs, mesh, state['trial_id']

def image_to_3d(image_path: str, seed: int = 0, randomize_seed: bool = True,
                ss_guidance_strength: float = 7.5, ss_sampling_steps: int = 12,
                slat_guidance_strength: float = 3.0, slat_sampling_steps: int = 12) -> Tuple[dict, str]:
    trial_id, _ = preprocess_image(image_path)
    if randomize_seed:
        seed = np.random.randint(0, MAX_SEED)
    outputs = pipeline.run(
        Image.open(f"{TMP_DIR}/{trial_id}.png"),
        seed=seed,
        formats=["gaussian", "mesh"],
        preprocess_image=False,
        sparse_structure_sampler_params={
            "steps": ss_sampling_steps,
            "cfg_strength": ss_guidance_strength,
        },
        slat_sampler_params={
            "steps": slat_sampling_steps,
            "cfg_strength": slat_guidance_strength,
        },
    )
    video = render_utils.render_video(outputs['gaussian'][0], num_frames=120)['color']
    video_geo = render_utils.render_video(outputs['mesh'][0], num_frames=120)['normal']
    video = [np.concatenate([video[i], video_geo[i]], axis=1) for i in range(len(video))]
    video_path = f"{TMP_DIR}/trellis-tost.mp4"
    imageio.mimsave(video_path, video, fps=15)
    state = pack_state(outputs['gaussian'][0], outputs['mesh'][0], "trellis-tost")
    return state, video_path

def extract_glb(state: dict, mesh_simplify: float = 0.95, texture_size: int = 1024) -> str:
    gs, mesh, trial_id = unpack_state(state)
    glb = postprocessing_utils.to_glb(gs, mesh, simplify=mesh_simplify, texture_size=texture_size, verbose=False)
    glb_path = f"{TMP_DIR}/{trial_id}.glb"
    glb.export(glb_path)
    return glb_path

with torch.inference_mode():
    pipeline = TrellisImageTo3DPipeline.from_pretrained("/content/model")
    pipeline.cuda()

print("✅ الموديل جاهز للاستخدام!")

/content/TRELLIS


RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-69b93a38-61dd42e232e4abf904c07e27;d6f1a215-39a7-44f0-958a-840e74cc9f50)

Repository Not Found for url: https://huggingface.co/ckpts/slat_dec_mesh_swin8_B_64l8m256c_fp16/resolve/main/.json.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

In [7]:
# 1. تحقق من الملف الفعلي
!head -5 /content/TRELLIS/trellis/modules/sparse/__init__.py

# 2. امسح كل ملفات __pycache__
!find /content/TRELLIS -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null
!find /content/TRELLIS -name "*.pyc" -delete 2>/dev/null

# 3. تأكد إن الملف يقول spconv
!sed -i "s/BACKEND = 'torchsparse'/BACKEND = 'spconv'/" /content/TRELLIS/trellis/modules/sparse/__init__.py
!head -5 /content/TRELLIS/trellis/modules/sparse/__init__.py

# 4. تحقق إن ملف .json موجود بالشكل الصحيح
!ls /content/model/ckpts/slat_dec_mesh_swin8_B_64l8m256c_fp16/

print("\n✅ تم مسح الـ cache وإصلاح الملفات")

from typing import *

BACKEND = 'spconv' 
DEBUG = False
ATTN = 'flash_attn'
from typing import *

BACKEND = 'spconv' 
DEBUG = False
ATTN = 'flash_attn'
model.safetensors

✅ تم مسح الـ cache وإصلاح الملفات


In [8]:
# 1. تحقق من بنية كل مجلدات الـ checkpoints
!echo "=== محتوى مجلد واحد كمثال ==="
!ls -la /content/model/ckpts/slat_dec_mesh_swin8_B_64l8m256c_fp16/

# 2. تحقق إن .json موجود في كل المجلدات
!echo ""
!echo "=== ملفات .json في المجلدات الفرعية ==="
!find /content/model/ckpts/*/  -name "*.json" -o -name ".json" 2>/dev/null

# 3. شف محتوى الـ .json
!echo ""
!echo "=== محتوى .json ==="
!cat /content/model/ckpts/slat_dec_mesh_swin8_B_64l8m256c_fp16/.json 2>/dev/null || echo "❌ ملف .json غير موجود!"

=== محتوى مجلد واحد كمثال ===
total 177656
drwxr-xr-x  2 root root      4096 Mar 17 11:04 .
drwxr-xr-x 10 root root      4096 Mar 17 11:04 ..
-rw-r--r--  1 root root       372 Mar 17 10:51 .json
-rw-r--r--  1 root root 181903412 Mar 17 10:51 model.safetensors

=== ملفات .json في المجلدات الفرعية ===
/content/model/ckpts/slat_dec_gs_swin8_B_64l8gs32_fp16/.json
/content/model/ckpts/slat_dec_mesh_swin8_B_64l8m256c_fp16/.json
/content/model/ckpts/slat_dec_rf_swin8_B_64l8r16_fp16/.json
/content/model/ckpts/slat_enc_swin8_B_64l8_fp16/.json
/content/model/ckpts/slat_flow_img_dit_L_64l8p2_fp16/.json
/content/model/ckpts/ss_dec_conv3d_16l8_fp16/.json
/content/model/ckpts/ss_enc_conv3d_16l8_fp16/.json
/content/model/ckpts/ss_flow_img_dit_L_16l8_fp16/.json

=== محتوى .json ===
{
    "name": "SLatMeshDecoder",
    "args": {
        "resolution": 64,
        "model_channels": 768,
        "latent_channels": 8,
        "num_blocks": 12,
        "num_heads": 12,
        "mlp_ratio": 4,
        "attn_

In [9]:
# ⚠️ مهم: هذي السطور لازم تكون أول شي
import os
os.environ['CUMM_DISABLE_JIT'] = '1'
os.environ['SPCONV_DISABLE_JIT'] = '1'
os.environ['ATTN_BACKEND'] = 'xformers'
os.environ['SPCONV_ALGO'] = 'native'

# مسح أي modules محملة سابقاً من trellis
import sys
mods_to_remove = [k for k in sys.modules if k.startswith('trellis')]
for m in mods_to_remove:
    del sys.modules[m]

%cd /content/TRELLIS

import numpy as np
import torch
import imageio
from typing import *
from PIL import Image
from easydict import EasyDict as edict
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.representations import Gaussian, MeshExtractResult
from trellis.utils import render_utils, postprocessing_utils

MAX_SEED = np.iinfo(np.int32).max
TMP_DIR = "/content"

def preprocess_image(image_path: str) -> Tuple[str, Image.Image]:
    trial_id = "trellis-tost"
    image = Image.open(image_path).convert("RGBA")
    processed_image = pipeline.preprocess_image(image)
    processed_image.save(f"{TMP_DIR}/{trial_id}.png")
    return trial_id, processed_image

def pack_state(gs: Gaussian, mesh: MeshExtractResult, trial_id: str) -> dict:
    return {
        'gaussian': {
            **gs.init_params,
            '_xyz': gs._xyz.cpu().numpy(),
            '_features_dc': gs._features_dc.cpu().numpy(),
            '_scaling': gs._scaling.cpu().numpy(),
            '_rotation': gs._rotation.cpu().numpy(),
            '_opacity': gs._opacity.cpu().numpy(),
        },
        'mesh': {
            'vertices': mesh.vertices.cpu().numpy(),
            'faces': mesh.faces.cpu().numpy(),
        },
        'trial_id': trial_id,
    }

def unpack_state(state: dict) -> Tuple[Gaussian, edict, str]:
    gs = Gaussian(
        aabb=state['gaussian']['aabb'],
        sh_degree=state['gaussian']['sh_degree'],
        mininum_kernel_size=state['gaussian']['mininum_kernel_size'],
        scaling_bias=state['gaussian']['scaling_bias'],
        opacity_bias=state['gaussian']['opacity_bias'],
        scaling_activation=state['gaussian']['scaling_activation'],
    )
    gs._xyz = torch.tensor(state['gaussian']['_xyz'], device='cuda')
    gs._features_dc = torch.tensor(state['gaussian']['_features_dc'], device='cuda')
    gs._scaling = torch.tensor(state['gaussian']['_scaling'], device='cuda')
    gs._rotation = torch.tensor(state['gaussian']['_rotation'], device='cuda')
    gs._opacity = torch.tensor(state['gaussian']['_opacity'], device='cuda')
    mesh = edict(
        vertices=torch.tensor(state['mesh']['vertices'], device='cuda'),
        faces=torch.tensor(state['mesh']['faces'], device='cuda'),
    )
    return gs, mesh, state['trial_id']

def image_to_3d(image_path: str, seed: int = 0, randomize_seed: bool = True,
                ss_guidance_strength: float = 7.5, ss_sampling_steps: int = 12,
                slat_guidance_strength: float = 3.0, slat_sampling_steps: int = 12) -> Tuple[dict, str]:
    trial_id, _ = preprocess_image(image_path)
    if randomize_seed:
        seed = np.random.randint(0, MAX_SEED)
    outputs = pipeline.run(
        Image.open(f"{TMP_DIR}/{trial_id}.png"),
        seed=seed,
        formats=["gaussian", "mesh"],
        preprocess_image=False,
        sparse_structure_sampler_params={
            "steps": ss_sampling_steps,
            "cfg_strength": ss_guidance_strength,
        },
        slat_sampler_params={
            "steps": slat_sampling_steps,
            "cfg_strength": slat_guidance_strength,
        },
    )
    video = render_utils.render_video(outputs['gaussian'][0], num_frames=120)['color']
    video_geo = render_utils.render_video(outputs['mesh'][0], num_frames=120)['normal']
    video = [np.concatenate([video[i], video_geo[i]], axis=1) for i in range(len(video))]
    video_path = f"{TMP_DIR}/trellis-tost.mp4"
    imageio.mimsave(video_path, video, fps=15)
    state = pack_state(outputs['gaussian'][0], outputs['mesh'][0], "trellis-tost")
    return state, video_path

def extract_glb(state: dict, mesh_simplify: float = 0.95, texture_size: int = 1024) -> str:
    gs, mesh, trial_id = unpack_state(state)
    glb = postprocessing_utils.to_glb(gs, mesh, simplify=mesh_simplify, texture_size=texture_size, verbose=False)
    glb_path = f"{TMP_DIR}/{trial_id}.glb"
    glb.export(glb_path)
    return glb_path

with torch.inference_mode():
    pipeline = TrellisImageTo3DPipeline.from_pretrained("/content/model")
    pipeline.cuda()

print("✅ الموديل جاهز للاستخدام!")

/content/TRELLIS
[SPARSE] Backend: torchsparse, Attention: xformers
[SPARSE][CONV] spconv algo: native


/content/TRELLIS/trellis/models/sparse_structure_vae.py:103: SyntaxWarning: invalid escape sequence '\m'
  Encoder for Sparse Structure (\mathcal{E}_S in the paper Sec. 3.3).
/content/TRELLIS/trellis/models/sparse_structure_vae.py:212: SyntaxWarning: invalid escape sequence '\m'
  Decoder for Sparse Structure (\mathcal{D}_S in the paper Sec. 3.3).


[ATTENTION] Using backend: xformers


RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-69b93b05-603bf4f6226a2aef065f1100;529d6d06-c85c-4c8b-b2ed-dd5f506ee24a)

Repository Not Found for url: https://huggingface.co/ckpts/slat_dec_mesh_swin8_B_64l8m256c_fp16/resolve/main/.json.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

In [2]:
# ============================================================
# الخلية 2: تشغيل الموديل
# ============================================================
import os
os.environ.setdefault('CUMM_DISABLE_JIT', '1')
os.environ.setdefault('SPCONV_DISABLE_JIT', '1')
os.environ.setdefault('ATTN_BACKEND', 'xformers')
os.environ.setdefault('SPCONV_ALGO', 'native')

%cd /content/TRELLIS

import numpy as np
import torch
import imageio
from typing import *
from PIL import Image
from easydict import EasyDict as edict
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.representations import Gaussian, MeshExtractResult
from trellis.utils import render_utils, postprocessing_utils

MAX_SEED = np.iinfo(np.int32).max
TMP_DIR = "/content"

def preprocess_image(image_path: str) -> Tuple[str, Image.Image]:
    trial_id = "trellis-tost"
    image = Image.open(image_path).convert("RGBA")
    processed_image = pipeline.preprocess_image(image)
    processed_image.save(f"{TMP_DIR}/{trial_id}.png")
    return trial_id, processed_image

def pack_state(gs: Gaussian, mesh: MeshExtractResult, trial_id: str) -> dict:
    return {
        'gaussian': {
            **gs.init_params,
            '_xyz': gs._xyz.cpu().numpy(),
            '_features_dc': gs._features_dc.cpu().numpy(),
            '_scaling': gs._scaling.cpu().numpy(),
            '_rotation': gs._rotation.cpu().numpy(),
            '_opacity': gs._opacity.cpu().numpy(),
        },
        'mesh': {
            'vertices': mesh.vertices.cpu().numpy(),
            'faces': mesh.faces.cpu().numpy(),
        },
        'trial_id': trial_id,
    }

def unpack_state(state: dict) -> Tuple[Gaussian, edict, str]:
    gs = Gaussian(
        aabb=state['gaussian']['aabb'],
        sh_degree=state['gaussian']['sh_degree'],
        mininum_kernel_size=state['gaussian']['mininum_kernel_size'],
        scaling_bias=state['gaussian']['scaling_bias'],
        opacity_bias=state['gaussian']['opacity_bias'],
        scaling_activation=state['gaussian']['scaling_activation'],
    )
    gs._xyz = torch.tensor(state['gaussian']['_xyz'], device='cuda')
    gs._features_dc = torch.tensor(state['gaussian']['_features_dc'], device='cuda')
    gs._scaling = torch.tensor(state['gaussian']['_scaling'], device='cuda')
    gs._rotation = torch.tensor(state['gaussian']['_rotation'], device='cuda')
    gs._opacity = torch.tensor(state['gaussian']['_opacity'], device='cuda')
    mesh = edict(
        vertices=torch.tensor(state['mesh']['vertices'], device='cuda'),
        faces=torch.tensor(state['mesh']['faces'], device='cuda'),
    )
    return gs, mesh, state['trial_id']

def image_to_3d(image_path: str, seed: int = 0, randomize_seed: bool = True,
                ss_guidance_strength: float = 7.5, ss_sampling_steps: int = 12,
                slat_guidance_strength: float = 3.0, slat_sampling_steps: int = 12) -> Tuple[dict, str]:
    trial_id, _ = preprocess_image(image_path)
    if randomize_seed:
        seed = np.random.randint(0, MAX_SEED)
    outputs = pipeline.run(
        Image.open(f"{TMP_DIR}/{trial_id}.png"),
        seed=seed,
        formats=["gaussian", "mesh"],
        preprocess_image=False,
        sparse_structure_sampler_params={
            "steps": ss_sampling_steps,
            "cfg_strength": ss_guidance_strength,
        },
        slat_sampler_params={
            "steps": slat_sampling_steps,
            "cfg_strength": slat_guidance_strength,
        },
    )
    video = render_utils.render_video(outputs['gaussian'][0], num_frames=120)['color']
    video_geo = render_utils.render_video(outputs['mesh'][0], num_frames=120)['normal']
    video = [np.concatenate([video[i], video_geo[i]], axis=1) for i in range(len(video))]
    video_path = f"{TMP_DIR}/trellis-tost.mp4"
    imageio.mimsave(video_path, video, fps=15)
    state = pack_state(outputs['gaussian'][0], outputs['mesh'][0], "trellis-tost")
    return state, video_path

def extract_glb(state: dict, mesh_simplify: float = 0.95, texture_size: int = 1024) -> str:
    gs, mesh, trial_id = unpack_state(state)
    glb = postprocessing_utils.to_glb(gs, mesh, simplify=mesh_simplify, texture_size=texture_size, verbose=False)
    glb_path = f"{TMP_DIR}/{trial_id}.glb"
    glb.export(glb_path)
    return glb_path

with torch.inference_mode():
    pipeline = TrellisImageTo3DPipeline.from_pretrained("/content/model")
    pipeline.cuda()

print("✅ الموديل جاهز للاستخدام!")

/content/TRELLIS
[SPARSE] Backend: torchsparse, Attention: xformers
[SPARSE][CONV] spconv algo: native
[ATTENTION] Using backend: xformers


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-69b938fc-23278fc13f6266b868259442;1b91be0f-1b49-400a-8323-72ed9d329811)

Repository Not Found for url: https://huggingface.co/ckpts/slat_dec_mesh_swin8_B_64l8m256c_fp16/resolve/main/.json.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

In [2]:
# ============================================================
# الخلية 2: تشغيل الموديل
# ============================================================
import os
os.environ['SPARSE_BACKEND'] = 'torchsparse'
os.environ['ATTN_BACKEND'] = 'xformers'
os.environ['SPCONV_ALGO'] = 'native'

%cd /content/TRELLIS

import numpy as np
import torch
import imageio
from typing import *
from PIL import Image
from easydict import EasyDict as edict
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.representations import Gaussian, MeshExtractResult
from trellis.utils import render_utils, postprocessing_utils

MAX_SEED = np.iinfo(np.int32).max
TMP_DIR = "/content"

# ... (باقي الدوال نفسها بدون تغيير)

with torch.inference_mode():
    pipeline = TrellisImageTo3DPipeline.from_pretrained("/content/model")
    pipeline.cuda()

print("✅ الموديل جاهز!")

/content/TRELLIS
[SPARSE] Backend: torchsparse, Attention: xformers


/content/TRELLIS/trellis/models/sparse_structure_vae.py:103: SyntaxWarning: invalid escape sequence '\m'
  Encoder for Sparse Structure (\mathcal{E}_S in the paper Sec. 3.3).
/content/TRELLIS/trellis/models/sparse_structure_vae.py:212: SyntaxWarning: invalid escape sequence '\m'
  Decoder for Sparse Structure (\mathcal{D}_S in the paper Sec. 3.3).


[SPARSE][CONV] spconv algo: native
[ATTENTION] Using backend: xformers


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-69b9344e-520d5a40387e990637d085a4;2cb952ba-48c4-4430-85f4-29141a78f773)

Repository Not Found for url: https://huggingface.co/ckpts/slat_dec_mesh_swin8_B_64l8m256c_fp16/resolve/main/.json.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

In [4]:
input_image = "/content/TRELLIS/T.png"
seed = 0
randomize_seed = True
ss_guidance_strength = 7.5
ss_sampling_steps = 12
slat_guidance_strength = 3.0
slat_sampling_steps = 12
mesh_simplify = 0.95
texture_size = 1024

state, video_path = image_to_3d(image_path=input_image,
                                seed=seed,
                                randomize_seed=randomize_seed,
                                ss_guidance_strength=ss_guidance_strength,
                                ss_sampling_steps=ss_sampling_steps,
                                slat_guidance_strength=slat_guidance_strength,
                                slat_sampling_steps=slat_sampling_steps)
glb_path = extract_glb(state=state, mesh_simplify=mesh_simplify, texture_size=texture_size)

FileNotFoundError: [Errno 2] No such file or directory: '/content/TRELLIS/T.png'

In [2]:
# ============================================================
# رفع صورة وتحويلها لـ 3D
# ============================================================

# الخطوة 1: رفع صورة
from google.colab import files
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print(f"📷 تم رفع: {image_path}")

# الخطوة 2: تحويل الصورة لـ 3D
state, video_path = image_to_3d(f"/content/{image_path}")
print(f"🎬 تم إنشاء الفيديو: {video_path}")

# الخطوة 3: تصدير ملف GLB (3D قابل للفتح)
glb_path = extract_glb(state)
print(f"📦 تم إنشاء ملف 3D: {glb_path}")

# الخطوة 4: تحميل الملفات
files.download(video_path)
files.download(glb_path)

print("\n✅ تم! افتح ملف .glb في https://3dviewer.net")

Saving 1800x1800_2964D28D-B1A2-467B-8DFB-AF18F02791FC.jpg-700.webp to 1800x1800_2964D28D-B1A2-467B-8DFB-AF18F02791FC.jpg-700.webp
📷 تم رفع: 1800x1800_2964D28D-B1A2-467B-8DFB-AF18F02791FC.jpg-700.webp


FileNotFoundError: [Errno 2] No such file or directory: '/content/1800x1800_2964D28D-B1A2-467B-8DFB-AF18F02791FC.jpg-700.webp'

In [3]:
# إصلاح المسار وتشغيل
import glob

# البحث عن الملف
found = glob.glob("/content/**/*.webp", recursive=True) + glob.glob("/content/**/*.png", recursive=True) + glob.glob("/content/**/*.jpg", recursive=True)
print("الملفات الموجودة:", found)

# أو ارفع من جديد مع إصلاح المسار
from google.colab import files
uploaded = files.upload()
image_name = list(uploaded.keys())[0]

# حفظ الملف بمسار معروف
import shutil
image_path = f"/content/input_image.png"
from PIL import Image
img = Image.open(image_name)
img.save(image_path)
print(f"📷 تم حفظ الصورة: {image_path}")

# تحويل لـ 3D
state, video_path = image_to_3d(image_path)
print(f"🎬 فيديو: {video_path}")

# ��صدير GLB
glb_path = extract_glb(state)
print(f"📦 ملف 3D: {glb_path}")

# تحميل
files.download(video_path)
files.download(glb_path)
print("\n✅ تم! افتح .glb في https://3dviewer.net")

الملفات الموجودة: ['/content/TRELLIS/1800x1800_2964D28D-B1A2-467B-8DFB-AF18F02791FC.jpg-700.webp', '/content/TRELLIS/assets/logo.webp', '/content/TRELLIS/assets/teaser.png', '/content/TRELLIS/assets/example_image/typical_misc_paper_machine.png', '/content/TRELLIS/assets/example_image/typical_building_maya_pyramid.png', '/content/TRELLIS/assets/example_image/typical_vehicle_pirate_ship.png', '/content/TRELLIS/assets/example_image/typical_misc_telephone.png', '/content/TRELLIS/assets/example_image/typical_misc_magicbook.png', '/content/TRELLIS/assets/example_image/typical_humanoid_dragonborn.png', '/content/TRELLIS/assets/example_image/typical_vehicle_biplane.png', '/content/TRELLIS/assets/example_image/typical_misc_workbench.png', '/content/TRELLIS/assets/example_image/T.png', '/content/TRELLIS/assets/example_image/typical_misc_fireplace.png', '/content/TRELLIS/assets/example_image/typical_vehicle_excavator.png', '/content/TRELLIS/assets/example_image/typical_creature_dragon.png', '/con

Saving 1800x1800_2964D28D-B1A2-467B-8DFB-AF18F02791FC.jpg-700.webp to 1800x1800_2964D28D-B1A2-467B-8DFB-AF18F02791FC.jpg-700 (1).webp
📷 تم حفظ الصورة: /content/input_image.png


Sampling:   0%|          | 0/12 [00:00<?, ?it/s]


AttributeError: module 'xformers.ops.fmha' has no attribute 'BlockDiagonalMask'

In [1]:
# أوقف واعمل Restart session ثم شغل هذا

import os, sys
os.environ['CUMM_DISABLE_JIT'] = '1'
os.environ['SPCONV_DISABLE_JIT'] = '1'
os.environ['ATTN_BACKEND'] = 'flash_attn'    # ← تغيّر لـ flash_attn
os.environ['SPCONV_ALGO'] = 'native'

# إصلاح الملفات
init_file = "/content/TRELLIS/trellis/modules/sparse/__init__.py"
with open(init_file, 'r') as f:
    content = f.read()
content = content.replace("BACKEND = 'torchsparse'", "BACKEND = 'spconv'")
with open(init_file, 'w') as f:
    f.write(content)

os.system("find /content/TRELLIS -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null")
os.system("find /content/TRELLIS -name '*.pyc' -delete 2>/dev/null")

sys.path.insert(0, '/content/TRELLIS')
os.chdir('/content/TRELLIS')

import numpy as np, torch, imageio
from typing import *
from PIL import Image
from easydict import EasyDict as edict
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.representations import Gaussian, MeshExtractResult
from trellis.utils import render_utils, postprocessing_utils

MAX_SEED = np.iinfo(np.int32).max
TMP_DIR = "/content"

def preprocess_image(image_path):
    trial_id = "trellis-tost"
    image = Image.open(image_path).convert("RGBA")
    processed_image = pipeline.preprocess_image(image)
    processed_image.save(f"{TMP_DIR}/{trial_id}.png")
    return trial_id, processed_image

def pack_state(gs, mesh, trial_id):
    return {
        'gaussian': {**gs.init_params, '_xyz': gs._xyz.cpu().numpy(), '_features_dc': gs._features_dc.cpu().numpy(),
            '_scaling': gs._scaling.cpu().numpy(), '_rotation': gs._rotation.cpu().numpy(), '_opacity': gs._opacity.cpu().numpy()},
        'mesh': {'vertices': mesh.vertices.cpu().numpy(), 'faces': mesh.faces.cpu().numpy()}, 'trial_id': trial_id}

def unpack_state(state):
    gs = Gaussian(aabb=state['gaussian']['aabb'], sh_degree=state['gaussian']['sh_degree'],
        mininum_kernel_size=state['gaussian']['mininum_kernel_size'], scaling_bias=state['gaussian']['scaling_bias'],
        opacity_bias=state['gaussian']['opacity_bias'], scaling_activation=state['gaussian']['scaling_activation'])
    gs._xyz = torch.tensor(state['gaussian']['_xyz'], device='cuda')
    gs._features_dc = torch.tensor(state['gaussian']['_features_dc'], device='cuda')
    gs._scaling = torch.tensor(state['gaussian']['_scaling'], device='cuda')
    gs._rotation = torch.tensor(state['gaussian']['_rotation'], device='cuda')
    gs._opacity = torch.tensor(state['gaussian']['_opacity'], device='cuda')
    mesh = edict(vertices=torch.tensor(state['mesh']['vertices'], device='cuda'), faces=torch.tensor(state['mesh']['faces'], device='cuda'))
    return gs, mesh, state['trial_id']

def image_to_3d(image_path, seed=0, randomize_seed=True, ss_guidance_strength=7.5, ss_sampling_steps=12, slat_guidance_strength=3.0, slat_sampling_steps=12):
    trial_id, _ = preprocess_image(image_path)
    if randomize_seed: seed = np.random.randint(0, MAX_SEED)
    outputs = pipeline.run(Image.open(f"{TMP_DIR}/{trial_id}.png"), seed=seed, formats=["gaussian", "mesh"], preprocess_image=False,
        sparse_structure_sampler_params={"steps": ss_sampling_steps, "cfg_strength": ss_guidance_strength},
        slat_sampler_params={"steps": slat_sampling_steps, "cfg_strength": slat_guidance_strength})
    video = render_utils.render_video(outputs['gaussian'][0], num_frames=120)['color']
    video_geo = render_utils.render_video(outputs['mesh'][0], num_frames=120)['normal']
    video = [np.concatenate([video[i], video_geo[i]], axis=1) for i in range(len(video))]
    video_path = f"{TMP_DIR}/trellis-tost.mp4"
    imageio.mimsave(video_path, video, fps=15)
    return pack_state(outputs['gaussian'][0], outputs['mesh'][0], "trellis-tost"), video_path

def extract_glb(state, mesh_simplify=0.95, texture_size=1024):
    gs, mesh, trial_id = unpack_state(state)
    glb = postprocessing_utils.to_glb(gs, mesh, simplify=mesh_simplify, texture_size=texture_size, verbose=False)
    glb_path = f"{TMP_DIR}/{trial_id}.glb"
    glb.export(glb_path)
    return glb_path

with torch.inference_mode():
    pipeline = TrellisImageTo3DPipeline.from_pretrained("/content/model")
    pipeline.cuda()
print("✅ الموديل جاهز!")

[SPARSE] Backend: spconv, Attention: flash_attn


/content/TRELLIS/trellis/models/sparse_structure_vae.py:103: SyntaxWarning: invalid escape sequence '\m'
  Encoder for Sparse Structure (\mathcal{E}_S in the paper Sec. 3.3).
/content/TRELLIS/trellis/models/sparse_structure_vae.py:212: SyntaxWarning: invalid escape sequence '\m'
  Decoder for Sparse Structure (\mathcal{D}_S in the paper Sec. 3.3).


[SPARSE][CONV] spconv algo: native
[ATTENTION] Using backend: flash_attn


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-69b93e20-0655f1cd3beb14303bde9f4f;0a4213e9-dd3a-49e5-94e4-132ebc900a3e)

Repository Not Found for url: https://huggingface.co/ckpts/ss_flow_img_dit_L_16l8_fp16/resolve/main/.json.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

In [2]:
from google.colab import files
uploaded = files.upload()
image_name = list(uploaded.keys())[0]
from PIL import Image
Image.open(image_name).save("/content/input_image.png")

state, video_path = image_to_3d("/content/input_image.png")
glb_path = extract_glb(state)
files.download(video_path)
files.download(glb_path)
print("✅ تم!")

Saving 1800x1800_2964D28D-B1A2-467B-8DFB-AF18F02791FC.jpg-700.webp to 1800x1800_2964D28D-B1A2-467B-8DFB-AF18F02791FC.jpg-700 (2).webp


NameError: name 'pipeline' is not defined

In [3]:
# ============================================================
# الخلية الوحيدة — تثبيت + تشغيل + واجهة
# ============================================================
import os
os.environ['CUMM_DISABLE_JIT'] = '1'
os.environ['SPCONV_DISABLE_JIT'] = '1'
os.environ['ATTN_BACKEND'] = 'xformers'
os.environ['SPCONV_ALGO'] = 'native'

# --- تثبيت الحزم الناقصة ---
print("⏳ جاري تثبيت الحزم...")
os.system("pip install -q cumm-cu124 spconv-cu124")
os.system("pip install -q flash-attn --no-build-isolation")

# --- إصلاح xformers BlockDiagonalMask ---
import xformers.ops.fmha as fmha
if not hasattr(fmha, 'BlockDiagonalMask'):
    from xformers.ops.fmha.attn_bias import BlockDiagonalMask
    fmha.BlockDiagonalMask = BlockDiagonalMask
    print("✅ تم إصلاح xformers BlockDiagonalMask")

# --- إصلاح ملف TRELLIS ---
init_file = "/content/TRELLIS/trellis/modules/sparse/__init__.py"
with open(init_file, 'r') as f:
    content = f.read()
content = content.replace("BACKEND = 'torchsparse'", "BACKEND = 'spconv'")
with open(init_file, 'w') as f:
    f.write(content)
os.system("find /content/TRELLIS -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null")
os.system("find /content/TRELLIS -name '*.pyc' -delete 2>/dev/null")

import sys
sys.path.insert(0, '/content/TRELLIS')
os.chdir('/content/TRELLIS')

# --- تحميل الموديل ---
print("⏳ جاري تحميل الموديل...")
import numpy as np, torch, imageio
from PIL import Image
from easydict import EasyDict as edict
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.representations import Gaussian, MeshExtractResult
from trellis.utils import render_utils, postprocessing_utils

with torch.inference_mode():
    pipeline = TrellisImageTo3DPipeline.from_pretrained("/content/model")
    pipeline.cuda()
print("✅ الموديل جاهز!")

# ============================================================
# واجهة Gradio
# ============================================================
os.system("pip install -q gradio")
import gradio as gr

MAX_SEED = np.iinfo(np.int32).max

def generate_3d(image, seed, randomize_seed, ss_steps, ss_cfg, slat_steps, slat_cfg):
    if image is None:
        return None, None, "❌ ارفع صورة أولاً"

    if randomize_seed:
        seed = np.random.randint(0, MAX_SEED)

    image = Image.fromarray(image).convert("RGBA")
    processed = pipeline.preprocess_image(image)

    with torch.inference_mode():
        outputs = pipeline.run(
            processed, seed=int(seed), formats=["gaussian", "mesh"], preprocess_image=False,
            sparse_structure_sampler_params={"steps": int(ss_steps), "cfg_strength": float(ss_cfg)},
            slat_sampler_params={"steps": int(slat_steps), "cfg_strength": float(slat_cfg)},
        )

    # فيديو
    video = render_utils.render_video(outputs['gaussian'][0], num_frames=120)['color']
    video_geo = render_utils.render_video(outputs['mesh'][0], num_frames=120)['normal']
    video = [np.concatenate([video[i], video_geo[i]], axis=1) for i in range(len(video))]
    video_path = "/content/output.mp4"
    imageio.mimsave(video_path, video, fps=15)

    # GLB
    gs = outputs['gaussian'][0]
    mesh = outputs['mesh'][0]
    glb = postprocessing_utils.to_glb(gs, mesh, simplify=0.95, texture_size=1024, verbose=False)
    glb_path = "/content/output.glb"
    glb.export(glb_path)

    return video_path, glb_path, f"✅ تم! Seed: {seed}"

with gr.Blocks(title="TRELLIS 3D Generator") as demo:
    gr.Markdown("# 🎨 TRELLIS — تحويل صورة إلى 3D")

    with gr.Row():
        with gr.Column():
            input_image = gr.Image(label="📷 ارفع صورة", type="numpy")
            seed = gr.Number(label="Seed", value=0)
            randomize_seed = gr.Checkbox(label="Random Seed", value=True)
            ss_steps = gr.Slider(1, 50, value=12, step=1, label="SS Steps")
            ss_cfg = gr.Slider(1, 15, value=7.5, step=0.5, label="SS CFG")
            slat_steps = gr.Slider(1, 50, value=12, step=1, label="SLAT Steps")
            slat_cfg = gr.Slider(1, 15, value=3.0, step=0.5, label="SLAT CFG")
            generate_btn = gr.Button("🚀 حوّل لـ 3D", variant="primary")

        with gr.Column():
            output_video = gr.Video(label="🎬 فيديو")
            output_glb = gr.File(label="📦 ملف GLB (حمّله وافتحه في 3dviewer.net)")
            status = gr.Textbox(label="الحالة")

    generate_btn.click(
        fn=generate_3d,
        inputs=[input_image, seed, randomize_seed, ss_steps, ss_cfg, slat_steps, slat_cfg],
        outputs=[output_video, output_glb, status]
    )

demo.launch(share=True, debug=True)

⏳ جاري تثبيت الحزم...
✅ تم إصلاح xformers BlockDiagonalMask
⏳ جاري تحميل الموديل...
[ATTENTION] Using backend: xformers


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


✅ الموديل جاهز!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://212b240cdd8106cb57.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Sampling:   0%|          | 0/12 [00:00<?, ?it/s]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1698, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 63, in run_sync
    return await get_async_back

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://212b240cdd8106cb57.gradio.live


In [1]:
# ============================================================
# الخلية الوحيدة — النسخة النهائية
# ============================================================
import os
os.environ['CUMM_DISABLE_JIT'] = '1'
os.environ['SPCONV_DISABLE_JIT'] = '1'
os.environ['ATTN_BACKEND'] = 'xformers'
os.environ['SPCONV_ALGO'] = 'native'

# --- تثبيت الحزم الناقصة ---
print("⏳ تثبيت الحزم...")
os.system("pip install -q cumm-cu124 spconv-cu124")

# --- إصلاح ملف TRELLIS ---
init_file = "/content/TRELLIS/trellis/modules/sparse/__init__.py"
with open(init_file, 'r') as f:
    content = f.read()
content = content.replace("BACKEND = 'torchsparse'", "BACKEND = 'spconv'")
with open(init_file, 'w') as f:
    f.write(content)
os.system("find /content/TRELLIS -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null")
os.system("find /content/TRELLIS -name '*.pyc' -delete 2>/dev/null")

# --- إصلاح xformers BlockDiagonalMask قبل أي import ---
import xformers.ops.fmha as fmha
if not hasattr(fmha, 'BlockDiagonalMask'):
    from xformers.ops.fmha.attn_bias import BlockDiagonalMask
    fmha.BlockDiagonalMask = BlockDiagonalMask
    print("✅ xformers BlockDiagonalMask fixed")

# --- إصلاح ملف attention ليستخدم xformers دائماً (وليس flash_attn) ---
attn_file = "/content/TRELLIS/trellis/modules/sparse/attention/full_attn.py"
with open(attn_file, 'r') as f:
    attn_code = f.read()

# نضيف fallback: إذا xformers ما فيه BlockDiagonalMask نستورده
patch = """
# --- PATCHED: Fix BlockDiagonalMask import ---
try:
    from xformers.ops.fmha import BlockDiagonalMask as _BDM
    import xformers.ops.fmha as _fmha
    if not hasattr(_fmha, 'BlockDiagonalMask'):
        _fmha.BlockDiagonalMask = _BDM
except:
    try:
        from xformers.ops.fmha.attn_bias import BlockDiagonalMask as _BDM
        import xformers.ops.fmha as _fmha
        if not hasattr(_fmha, 'BlockDiagonalMask'):
            _fmha.BlockDiagonalMask = _BDM
    except:
        pass
# --- END PATCH ---
"""

if '# --- PATCHED' not in attn_code:
    attn_code = patch + attn_code
    with open(attn_file, 'w') as f:
        f.write(attn_code)
    print("✅ attention file patched")

# --- تحميل الموديل ---
import sys
sys.path.insert(0, '/content/TRELLIS')
os.chdir('/content/TRELLIS')

print("⏳ تحميل الموديل...")
import numpy as np, torch, imageio
from PIL import Image
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import render_utils, postprocessing_utils

with torch.inference_mode():
    pipeline = TrellisImageTo3DPipeline.from_pretrained("/content/model")
    pipeline.cuda()
print("✅ الموديل جاهز!")

# ============================================================
# واجهة Gradio
# ============================================================
os.system("pip install -q gradio")
import gradio as gr

def generate_3d(image, seed, randomize_seed, ss_steps, ss_cfg, slat_steps, slat_cfg):
    if image is None:
        return None, None, "❌ ارفع صورة أولاً"
    if randomize_seed:
        seed = np.random.randint(0, np.iinfo(np.int32).max)

    image = Image.fromarray(image).convert("RGBA")
    processed = pipeline.preprocess_image(image)

    with torch.inference_mode():
        outputs = pipeline.run(
            processed, seed=int(seed), formats=["gaussian", "mesh"], preprocess_image=False,
            sparse_structure_sampler_params={"steps": int(ss_steps), "cfg_strength": float(ss_cfg)},
            slat_sampler_params={"steps": int(slat_steps), "cfg_strength": float(slat_cfg)},
        )

    video = render_utils.render_video(outputs['gaussian'][0], num_frames=120)['color']
    video_geo = render_utils.render_video(outputs['mesh'][0], num_frames=120)['normal']
    video = [np.concatenate([video[i], video_geo[i]], axis=1) for i in range(len(video))]
    imageio.mimsave("/content/output.mp4", video, fps=15)

    glb = postprocessing_utils.to_glb(outputs['gaussian'][0], outputs['mesh'][0], simplify=0.95, texture_size=1024, verbose=False)
    glb.export("/content/output.glb")

    return "/content/output.mp4", "/content/output.glb", f"✅ تم! Seed: {seed}"

with gr.Blocks(title="TRELLIS 3D") as demo:
    gr.Markdown("# 🎨 TRELLIS — تحويل صورة إلى 3D")
    with gr.Row():
        with gr.Column():
            input_image = gr.Image(label="📷 ارفع صورة", type="numpy")
            seed = gr.Number(label="Seed", value=0)
            randomize_seed = gr.Checkbox(label="Random Seed", value=True)
            ss_steps = gr.Slider(1, 50, value=12, step=1, label="SS Steps")
            ss_cfg = gr.Slider(1, 15, value=7.5, step=0.5, label="SS CFG")
            slat_steps = gr.Slider(1, 50, value=12, step=1, label="SLAT Steps")
            slat_cfg = gr.Slider(1, 15, value=3.0, step=0.5, label="SLAT CFG")
            btn = gr.Button("🚀 حوّل لـ 3D", variant="primary")
        with gr.Column():
            video = gr.Video(label="🎬 فيديو")
            glb = gr.File(label="📦 GLB — حمّله وافتحه في 3dviewer.net")
            status = gr.Textbox(label="الحالة")
    btn.click(generate_3d, [input_image, seed, randomize_seed, ss_steps, ss_cfg, slat_steps, slat_cfg], [video, glb, status])

demo.launch(share=True, debug=True)

⏳ تثبيت الحزم...
✅ xformers BlockDiagonalMask fixed
✅ attention file patched
⏳ تحميل الموديل...
[SPARSE] Backend: spconv, Attention: xformers


/content/TRELLIS/trellis/models/sparse_structure_vae.py:103: SyntaxWarning: invalid escape sequence '\m'
  Encoder for Sparse Structure (\mathcal{E}_S in the paper Sec. 3.3).
/content/TRELLIS/trellis/models/sparse_structure_vae.py:212: SyntaxWarning: invalid escape sequence '\m'
  Decoder for Sparse Structure (\mathcal{D}_S in the paper Sec. 3.3).


[SPARSE][CONV] spconv algo: native
[ATTENTION] Using backend: xformers


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


✅ الموديل جاهز!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://78c7856a4f15c74767.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Sampling: 100%|██████████| 12/12 [00:06<00:00,  1.94it/s]
Rendering: 0it [00:00, ?it/s]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1698, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 63, in r

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://78c7856a4f15c74767.gradio.live


In [ ]:
# ============================================================
# النسخة النهائية المكتملة
# ============================================================
import os
os.environ['CUMM_DISABLE_JIT'] = '1'
os.environ['SPCONV_DISABLE_JIT'] = '1'
os.environ['ATTN_BACKEND'] = 'xformers'
os.environ['SPCONV_ALGO'] = 'native'

# --- تثبيت الحزم ---
print("⏳ تثبيت الحزم...")
os.system("pip install -q cumm-cu124 spconv-cu124")

# النسخة المعدّلة من diff-gaussian-rasterization اللي تدعم kernel_size
os.system("pip uninstall diff-gaussian-rasterization -y -q 2>/dev/null")
os.system("pip install -q --no-build-isolation git+https://github.com/JeffreyXiang/diff-gaussian-rasterization")

# --- إصلاح ملف TRELLIS ---
init_file = "/content/TRELLIS/trellis/modules/sparse/__init__.py"
with open(init_file, 'r') as f:
    content = f.read()
content = content.replace("BACKEND = 'torchsparse'", "BACKEND = 'spconv'")
with open(init_file, 'w') as f:
    f.write(content)
os.system("find /content/TRELLIS -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null")
os.system("find /content/TRELLIS -name '*.pyc' -delete 2>/dev/null")

# --- إصلاح xformers ---
import xformers.ops.fmha as fmha
if not hasattr(fmha, 'BlockDiagonalMask'):
    from xformers.ops.fmha.attn_bias import BlockDiagonalMask
    fmha.BlockDiagonalMask = BlockDiagonalMask
    print("✅ xformers fixed")

# --- Patch ملف attention ---
attn_file = "/content/TRELLIS/trellis/modules/sparse/attention/full_attn.py"
with open(attn_file, 'r') as f:
    attn_code = f.read()
patch = """
# --- PATCHED ---
try:
    from xformers.ops.fmha.attn_bias import BlockDiagonalMask as _BDM
    import xformers.ops.fmha as _fmha
    if not hasattr(_fmha, 'BlockDiagonalMask'):
        _fmha.BlockDiagonalMask = _BDM
except: pass
# --- END ---
"""
if '# --- PATCHED' not in attn_code:
    attn_code = patch + attn_code
    with open(attn_file, 'w') as f:
        f.write(attn_code)
    print("✅ attention patched")

# --- تحميل الموديل ---
import sys
sys.path.insert(0, '/content/TRELLIS')
os.chdir('/content/TRELLIS')

print("⏳ تحميل الموديل...")
import numpy as np, torch, imageio
from PIL import Image
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import render_utils, postprocessing_utils

with torch.inference_mode():
    pipeline = TrellisImageTo3DPipeline.from_pretrained("/content/model")
    pipeline.cuda()
print("✅ الموديل جاهز!")

# ============================================================
# واجهة Gradio
# ============================================================
os.system("pip install -q gradio")
import gradio as gr

def generate_3d(image, seed, randomize_seed, ss_steps, ss_cfg, slat_steps, slat_cfg):
    if image is None:
        return None, None, "❌ ارفع صورة أولاً"
    if randomize_seed:
        seed = np.random.randint(0, np.iinfo(np.int32).max)

    image = Image.fromarray(image).convert("RGBA")
    processed = pipeline.preprocess_image(image)

    with torch.inference_mode():
        outputs = pipeline.run(
            processed, seed=int(seed), formats=["gaussian", "mesh"], preprocess_image=False,
            sparse_structure_sampler_params={"steps": int(ss_steps), "cfg_strength": float(ss_cfg)},
            slat_sampler_params={"steps": int(slat_steps), "cfg_strength": float(slat_cfg)},
        )

    video = render_utils.render_video(outputs['gaussian'][0], num_frames=120)['color']
    video_geo = render_utils.render_video(outputs['mesh'][0], num_frames=120)['normal']
    video = [np.concatenate([video[i], video_geo[i]], axis=1) for i in range(len(video))]
    imageio.mimsave("/content/output.mp4", video, fps=15)

    glb = postprocessing_utils.to_glb(outputs['gaussian'][0], outputs['mesh'][0], simplify=0.95, texture_size=1024, verbose=False)
    glb.export("/content/output.glb")

    return "/content/output.mp4", "/content/output.glb", f"✅ تم! Seed: {seed}"

with gr.Blocks(title="TRELLIS 3D") as demo:
    gr.Markdown("# 🎨 TRELLIS — تحويل صورة إلى 3D")
    with gr.Row():
        with gr.Column():
            input_image = gr.Image(label="�� ارفع صورة", type="numpy")
            seed = gr.Number(label="Seed", value=0)
            randomize_seed = gr.Checkbox(label="Random Seed", value=True)
            ss_steps = gr.Slider(1, 50, value=12, step=1, label="SS Steps")
            ss_cfg = gr.Slider(1, 15, value=7.5, step=0.5, label="SS CFG")
            slat_steps = gr.Slider(1, 50, value=12, step=1, label="SLAT Steps")
            slat_cfg = gr.Slider(1, 15, value=3.0, step=0.5, label="SLAT CFG")
            btn = gr.Button("🚀 حوّل لـ 3D", variant="primary")
        with gr.Column():
            video = gr.Video(label="🎬 فيديو")
            glb = gr.File(label="📦 GLB — حمّله وافتحه في 3dviewer.net")
            status = gr.Textbox(label="الحالة")
    btn.click(generate_3d, [input_image, seed, randomize_seed, ss_steps, ss_cfg, slat_steps, slat_cfg], [video, glb, status])

demo.launch(share=True, debug=True)

⏳ تثبيت الحزم...
✅ xformers fixed
⏳ تحميل الموديل...
[SPARSE] Backend: spconv, Attention: xformers


/content/TRELLIS/trellis/models/sparse_structure_vae.py:103: SyntaxWarning: invalid escape sequence '\m'
  Encoder for Sparse Structure (\mathcal{E}_S in the paper Sec. 3.3).
/content/TRELLIS/trellis/models/sparse_structure_vae.py:212: SyntaxWarning: invalid escape sequence '\m'
  Decoder for Sparse Structure (\mathcal{D}_S in the paper Sec. 3.3).


[SPARSE][CONV] spconv algo: native
[ATTENTION] Using backend: xformers


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


✅ الموديل جاهز!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://332beefa742caa045e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Sampling: 100%|██████████| 12/12 [00:05<00:00,  2.28it/s]
Rendering: 0it [00:00, ?it/s]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1698, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 63, in r

In [ ]:
# تثبيت diff-gaussian-rasterization (النسخة المعدّلة لـ TRELLIS)
!pip install --no-build-isolation git+https://github.com/JeffreyXiang/diff-gaussian-rasterization 2>&1 | tail -5

# تحقق
!python -c "from diff_gaussian_rasterization import GaussianRasterizer; print('✅ diff_gaussian_rasterization OK')"